In [ ]:
import pandas as pd
from pathlib import Path

BASE = Path(__file__).parent if '__file__' in dir() else Path().resolve()

df = pd.read_csv(
    BASE / "../../../data/processed/2025/datos_limpios.csv",
    low_memory=False
)

# 1. Producción total por pozo en el año
print("=== TOP 10 POZOS POR PRODUCCIÓN DE PETRÓLEO ===")
por_pozo = df.groupby(['idpozo', 'empresa', 'provincia', 'tipo_de_recurso'])[
    ['prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'iny_gas', 'tef']
].sum()
por_pozo = por_pozo.sort_values('prod_pet', ascending=False)
print(por_pozo.head(10))

# 2. Eficiencia: relación producción vs inyección
print("\n=== EFICIENCIA DE POZOS ===")
por_pozo['inyeccion_total'] = por_pozo['iny_agua'] + por_pozo['iny_gas']
por_pozo['eficiencia_pet'] = por_pozo.apply(
    lambda x: x['prod_pet'] / x['inyeccion_total'] 
    if x['inyeccion_total'] > 0 else None, axis=1
)

pozos_con_inyeccion = por_pozo[
    (por_pozo['prod_pet'] > 0) & 
    (por_pozo['inyeccion_total'] > 0)
].copy()

print(f"Pozos que producen y usan inyección: {len(pozos_con_inyeccion)}")
print("\nTop 10 más eficientes:")
print(pozos_con_inyeccion.sort_values('eficiencia_pet', ascending=False).head(10))
print("\nTop 10 menos eficientes (más costosos):")
print(pozos_con_inyeccion.sort_values('eficiencia_pet', ascending=True).head(10))

# 3. Declive de producción
# Ojo: 2025 solo tiene 11 meses, se excluye mes 12
print("\n=== DECLIVE DE PRODUCCIÓN ===")
declive = df.groupby(['idpozo', 'mes'])['prod_pet'].sum().unstack(fill_value=0)

declive['primer_semestre'] = declive[[1,2,3,4,5,6]].sum(axis=1)
# Solo hasta mes 11 porque diciembre no está en el dataset
declive['segundo_semestre'] = declive[[7,8,9,10,11]].sum(axis=1)
declive['variacion'] = (
    (declive['segundo_semestre'] - declive['primer_semestre']) / 
    declive['primer_semestre'].replace(0, float('nan')) * 100
).round(2)

pozos_en_declive = declive[declive['variacion'] < -30].sort_values('variacion')
pozos_creciendo = declive[declive['variacion'] > 30].sort_values('variacion', ascending=False)

print(f"Pozos con caída mayor al 30%: {len(pozos_en_declive)}")
print(f"Pozos con crecimiento mayor al 30%: {len(pozos_creciendo)}")
print("\nTop 10 pozos en mayor declive:")
print(pozos_en_declive.head(10)[['primer_semestre', 'segundo_semestre', 'variacion']])
print("\nTop 10 pozos con mayor crecimiento:")
print(pozos_creciendo.head(10)[['primer_semestre', 'segundo_semestre', 'variacion']])

# 4. Guardar resultados para Power BI
por_pozo.reset_index().to_csv(
    BASE / "../../../data/processed/2025/analisis_pozos.csv",
    index=False
)
declive.reset_index().to_csv(
    BASE / "../../../data/processed/2025/declive_pozos.csv",
    index=False
)
print("\nArchivos guardados correctamente")

=== TOP 10 POZOS POR PRODUCCIÓN DE PETRÓLEO ===
                                                                         prod_pet  \
idpozo empresa                           provincia tipo_de_recurso                  
164941 VISTA ENERGY ARGENTINA SAU        Neuquén   NO CONVENCIONAL  148688.296000   
163633 PLUSPETROL CUENCA NEUQUINA S.R.L. Neuquén   NO CONVENCIONAL   75321.940000   
165607 PAN AMERICAN ENERGY SL            Neuquén   NO CONVENCIONAL   71158.529000   
163631 PLUSPETROL CUENCA NEUQUINA S.R.L. Neuquén   NO CONVENCIONAL   70961.730000   
165825 TOTAL AUSTRAL S.A.                Neuquén   NO CONVENCIONAL   68018.322663   
165609 PAN AMERICAN ENERGY SL            Neuquén   NO CONVENCIONAL   67946.576000   
165824 TOTAL AUSTRAL S.A.                Neuquén   NO CONVENCIONAL   66442.031997   
166149 PETROLERA EL TREBOL S.A.          Rio Negro NO CONVENCIONAL   65915.070000   
163632 PLUSPETROL CUENCA NEUQUINA S.R.L. Neuquén   NO CONVENCIONAL   64509.470000   
166203 YPF S.A.  